In [8]:
 !df -h /

Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme0n1p1  105G   89G   17G  84% /


In [2]:
%cd /home/ec2-user/SageMaker/Amazon_ML_GMNR

/home/ec2-user/SageMaker/Amazon_ML_GMNR


In [6]:
!git --version

git version 2.50.1


In [3]:
!git add -A

In [ ]:
!pip install rapidfuzz datasketch lightgbm "scikit-learn>=1.3" pandas numpy matplotlib --quiet
print("✓ Installation complete")

In [6]:
!git commit -m "Update project files"

[main cbebdbb] Update project files
 2 files changed, 184 insertions(+), 77 deletions(-)


In [10]:
import rapidfuzz, datasketch, lightgbm as lgb, sklearn
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import sagemaker, boto3

versions = {
    "rapidfuzz": rapidfuzz.__version__,
    "datasketch": datasketch.__version__,
    "lightgbm": lgb.__version__,
    "sklearn": sklearn.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "sagemaker": sagemaker.__version__,
}
for lib, v in versions.items():
    print(f"  ✓ {lib}: {v}")

  ✓ rapidfuzz: 3.14.5
  ✓ datasketch: 2.0.0
  ✓ lightgbm: 4.7.0
  ✓ sklearn: 1.7.2
  ✓ pandas: 2.3.3
  ✓ numpy: 2.2.6
  ✓ sagemaker: 2.257.6


In [10]:
import subprocess
import getpass
import os

username = input("GitHub username: ")
token = getpass.getpass("GitHub Personal Access Token: ")

# Set the remote URL without embedding credentials
subprocess.run(
    ["git", "remote", "set-url", "origin",
     "https://github.com/Gmj080305/Amazon_ML_GMNR.git"],
    check=True
)

# Provide credentials to Git through a temporary askpass script
askpass_script = "/tmp/git_askpass.sh"

with open(askpass_script, "w") as f:
    f.write("""#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "$GITHUB_USERNAME" ;;
  *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
  *) exit 1 ;;
esac
""")

os.chmod(askpass_script, 0o700)

env = os.environ.copy()
env["GIT_ASKPASS"] = askpass_script
env["GIT_TERMINAL_PROMPT"] = "0"
env["GITHUB_USERNAME"] = username
env["GITHUB_TOKEN"] = token

try:
    result = subprocess.run(
        ["git", "push", "origin", "main"],
        env=env,
        text=True,
        capture_output=True
    )

    print(result.stdout)
    print(result.stderr)

    if result.returncode == 0:
        print("Push successful!")
    else:
        print("Push failed. Check the error above.")

finally:
    del env
    del token
    del username

    if os.path.exists(askpass_script):
        os.remove(askpass_script)

GitHub username:  Gmj080305
GitHub Personal Access Token:  ········



error: RPC failed; HTTP 408 curl 22 The requested URL returned error: 408
send-pack: unexpected disconnect while reading sideband packet
fatal: the remote end hung up unexpectedly
Everything up-to-date

Push failed. Check the error above.


In [12]:
!git remote -v

origin	https://github.com/Gmj080305/Amazon_ML_GMNR.git (fetch)
origin	https://github.com/Gmj080305/Amazon_ML_GMNR.git (push)


In [9]:
%%writefile /home/ec2-user/SageMaker/Amazon_ML_GMNR/.gitignore

# ==================================================
# Large datasets (100 MB and above)
# ==================================================

# Ignore all files inside data/ by default
data/*

# Keep the data directory structure documented
!data/.gitkeep
!data/README.md

# ==================================================
# Python artifacts
# ==================================================

__pycache__/
*.py[cod]
*.pyo
*.pyd
.Python
*.egg-info/
dist/
build/

# ==================================================
# Jupyter Notebook
# ==================================================

.ipynb_checkpoints/
*/.ipynb_checkpoints/

# ==================================================
# Machine learning outputs
# ==================================================

*.pt
*.pth
*.ckpt
*.h5
*.hdf5
*.onnx
*.safetensors
*.pkl
*.pickle
*.joblib

# ==================================================
# Logs and temporary files
# ==================================================

*.log
*.tmp
.DS_Store

Overwriting /home/ec2-user/SageMaker/Amazon_ML_GMNR/.gitignore


In [5]:
session = sagemaker.Session()
bucket  = session.default_bucket()
region  = boto3.Session().region_name
role    = sagemaker.get_execution_role()
PREFIX  = "entity-resolution-challenge"

print(f"Bucket : s3://{bucket}/{PREFIX}/")
print(f"Region : {region}")
print(f"Role   : {role[:60]}...")

Bucket : s3://sagemaker-ap-southeast-2-725335003020/entity-resolution-challenge/
Region : ap-southeast-2
Role   : arn:aws:iam::725335003020:role/service-role/AmazonSageMaker-...


In [6]:
import os

for d in ["src", "output", "data/train", "data/test", "notebooks"]:
    os.makedirs(d, exist_ok=True)

for fname in ["normalize.py", "block.py", "features.py",
              "train.py", "predict.py", "evaluate.py"]:
    fpath = f"src/{fname}"
    if not os.path.exists(fpath):
        open(fpath, "w").write(f"# {fname} — entity resolution pipeline\n")

print("✓ Directory structure ready")
!find src data output -type f | sort

✓ Directory structure ready
src/block.py
src/evaluate.py
src/features.py
src/normalize.py
src/predict.py
src/train.py
